# Phase 1: YOLO Baseline Models & Board-Level Leakage Prevention
### Capstone Project: Automated Optical Inspection (AOI) on Bare PCBs
**Hardware:** Windows 10/11 | NVIDIA RTX 4090 (24GB VRAM)  
**Models:** YOLO26s, YOLO26m, YOLO11s (Control)  
**Goals:**
1. Establish a leak-free board-level split (preventing train/test crop leakage from the same PCB).
2. Train YOLO26s and YOLO26m with RTX 4090 optimizations (TF32, batch=32, FP16 AMP).
3. Evaluate against YOLO11s control on strict mAP@50-95 and per-class recall (especially *short* and *open_circuit*).
4. Run input resolution ablation (640 vs 800 vs 1024) to evaluate thin-defect detection.


## 1. Environment & CUDA Setup
Configure PyTorch for NVIDIA Ada Lovelace RTX 4090 with TF32 Tensor Cores.


In [ ]:
import os
import sys
import torch

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Dedicated VRAM: {vram:.2f} GB")
    # Enable TF32 for high throughput on RTX 4090
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    print("[+] TF32 Acceleration Enabled.")
else:
    print("[!] Warning: CUDA is not active.")


## 2. Board-Level Stratified Dataset Splitting
To avoid inflated scores from near-duplicate crops of the same board, we partition strictly by board ID.


In [ ]:
import re
import random
from collections import defaultdict
from pathlib import Path

DATASET_ROOT = Path("./dataset").resolve()
SPLIT_OUT_DIR = Path("./splits/phase1").resolve()

def create_board_level_split(data_root: Path, out_dir: Path, train_ratio=0.8, val_ratio=0.1, seed=42):
    random.seed(seed)
    img_exts = {".jpg", ".jpeg", ".png", ".bmp"}
    all_imgs = [p for p in data_root.rglob("*") if p.suffix.lower() in img_exts]
    print(f"Discovered {len(all_imgs)} images in dataset.")
    
    board_groups = defaultdict(list)
    for p in all_imgs:
        m = re.match(r"^(\d+)_", p.stem)
        board_id = m.group(1) if m else p.stem.split("_")[0]
        board_groups[board_id].append(p)
        
    print(f"Grouped into {len(board_groups)} distinct boards.")
    
    boards = list(board_groups.keys())
    random.shuffle(boards)
    
    n_total = len(all_imgs)
    n_train = int(n_total * train_ratio)
    n_val = int(n_total * val_ratio)
    
    splits = {"train": [], "val": [], "test": []}
    for b in boards:
        dest = "train"
        if len(splits["val"]) < n_val:
            dest = "val"
        elif len(splits["test"]) < (n_total - n_train - n_val):
            dest = "test"
        splits[dest].extend(board_groups[b])
        
    out_dir.mkdir(parents=True, exist_ok=True)
    for s_name, files in splits.items():
        with open(out_dir / f"{s_name}.txt", "w") as f:
            for item in files:
                f.write(f"{item.as_posix()}\n")
        print(f"  {s_name}: {len(files)} images")
        
    yaml_text = f"""path: {out_dir.resolve().as_posix()}
train: train.txt
val: val.txt
test: test.txt

names:
  0: mouse_bite
  1: spur
  2: missing_hole
  3: short
  4: open_circuit
  5: spurious_copper
"""
    yaml_file = out_dir / "pcb_phase1.yaml"
    yaml_file.write_text(yaml_text, encoding="utf-8")
    print(f"[+] Dataset YAML ready: {yaml_file}")
    return yaml_file

YAML_FILE = create_board_level_split(DATASET_ROOT, SPLIT_OUT_DIR)


## 3. Training Baseline Models (YOLO26s, YOLO26m, YOLO11s)
Run standardized fine-tuning under identical hyperparameter conditions.


In [ ]:
from ultralytics import YOLO

TRAIN_CONFIG = {
    "data": str(YAML_FILE),
    "epochs": 100,
    "imgsz": 640,
    "batch": 32,          # High-throughput batch size for 24GB VRAM
    "device": 0,
    "workers": 8,
    "amp": True,
    "patience": 25,
    "save": True,
    "seed": 42
}

models_to_train = [
    ("yolo26s.pt", "YOLO26s_Phase1"),
    ("yolo26m.pt", "YOLO26m_Phase1"),
    ("yolo11s.pt", "YOLO11s_Control")
]

results = {}
for ckpt, run_name in models_to_train:
    print(f"\n{'='*50}\nStarting: {run_name} (Checkpoint: {ckpt})\n{'='*50}")
    try:
        model = YOLO(ckpt)
    except Exception as e:
        print(f"[!] Checkpoint {ckpt} not found directly, falling back to yolo11 alternative: {e}")
        model = YOLO("yolo11s.pt")
        
    res = model.train(
        project="Phase1_YOLO_Baselines",
        name=run_name,
        **TRAIN_CONFIG
    )
    results[run_name] = res


## 4. Rigorous Evaluation on the Frozen Test Set
Compute mAP@50, mAP@50-95, and per-class recall specifically for catastrophic defects (*open_circuit*, *short*).


In [ ]:
import pandas as pd
from tabulate import tabulate

CLASSES = ["mouse_bite", "spur", "missing_hole", "short", "open_circuit", "spurious_copper"]

eval_summary = []
for _, run_name in models_to_train:
    best_weights = Path("Phase1_YOLO_Baselines") / run_name / "weights" / "best.pt"
    if not best_weights.exists():
        continue
    print(f"[*] Evaluating {run_name}...")
    model = YOLO(best_weights)
    val_res = model.val(data=str(YAML_FILE), split="test", imgsz=640, device=0)
    
    entry = {
        "Model": run_name,
        "mAP@50": round(float(val_res.box.map50), 4),
        "mAP@50-95": round(float(val_res.box.map), 4),
    }
    for idx, c_name in enumerate(CLASSES):
        entry[f"Recall_{c_name}"] = round(float(val_res.box.r[idx]), 4) if idx < len(val_res.box.r) else 0.0
    eval_summary.append(entry)

df_phase1 = pd.DataFrame(eval_summary)
display(df_phase1)


## 5. Ablation: Input Resolution Scaling (640 vs 800 vs 1024)
Evaluate whether increasing input resolution resolves thin defects like hair-line spurs and trace opens.


In [ ]:
# Resolution ablation on the winning Phase 1 model
winning_ckpt = "Phase1_YOLO_Baselines/YOLO26s_Phase1/weights/best.pt"

if Path(winning_ckpt).exists():
    resolution_results = []
    model_eval = YOLO(winning_ckpt)
    for res in [640, 800, 1024]:
        print(f"[*] Testing resolution {res}x{res}...")
        metrics = model_eval.val(data=str(YAML_FILE), split="test", imgsz=res, device=0)
        resolution_results.append({
            "Resolution": f"{res}x{res}",
            "mAP@50": round(float(metrics.box.map50), 4),
            "mAP@50-95": round(float(metrics.box.map), 4),
            "Recall (Open Circuit)": round(float(metrics.box.r[4]), 4),
            "Recall (Spur)": round(float(metrics.box.r[1]), 4),
        })
    df_res = pd.DataFrame(resolution_results)
    display(df_res)
